# Suite BLD — Schema builders as visitors of their meta-schemas

`Builders` holds the meta-schemas (`Builders.Meta`), in which every schema object is a reference object and every
reference between schema objects is a relation, and the builders over them: each a writable `Visitors.OfObject`, with
a member for each property and adjacency of its meta-schema, and VISITOR_PATTERN.md's shorthands. Schema objects and
data are built in a store, `store.moduleName.[...moduleName].schemaName(...)`. See docs/META_SCHEMAS.md, Plan.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Paths, Plain, Schemas as S
from mbse.Schemas.Framework.Errors import ReadOnlyError
from support import raises


def person(txn):
    """crm.Person, who knows other persons since a year, typed by the store's basic scalars."""
    S = txn.Schemas
    return txn.uid("bld.person/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Knows": S.OfRelation})
        .spec([
            lambda _: _.Person.name("crm.Person").ref(True).properties({"given_name": S.OfStr, "age": S.OfInt})
            .adjacencies([
                lambda a: a.name("acquaintances").relation([_.Knows]).me("knower"),
                lambda a: a.relation([_.Knows]).me("known"),
            ]),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")])
            .properties({"since": S.OfInt}).uniques("since"),
        ])]))])

## BLD-01 · The meta-schemas: reference object schemas, and relations between them, each valid

In [ ]:
plain = B.Store()  # a store is made holding its own plain schemas of schemas, and its own basic scalars
for name in plain.names():
    if name.startswith("Schemas."):
        assert plain.registered(name).validate() == [], (name, plain.registered(name).validate())
OfObject, HasProperty, HasSchema = (path.as_schema() for path in (plain.Schemas.OfObject, plain.Schemas.HasProperty,
                                                                plain.Schemas.HasSchema))
assert OfObject.ref and list(OfObject.properties) == ["name", "description", "ref", "singleton"]
assert list(OfObject.adjacencies) == ["parameters", "adjacencies", "properties", "refines"]
assert list(HasProperty.entry.properties) == ["name"]  # a member's name is its entry's
assert set(HasProperty.uniques) == {frozenset({"owner", "name"}), frozenset({"property"})}  # one owner, one name
assert HasSchema.uniques == (frozenset({"schema"}),) and plain.Schemas.AnyOf.as_schema().uniques == ()
assert all(link.schema is None for link in HasSchema.entry.links.values())  # follows from the adjacencies
assert ("Schemas.HasSchema", "schema") in [(a.relation.name, a.me) for a in plain.Schemas.OfUnion.as_schema().anonymous]  # a type
assert "refines" not in plain.Schemas.OfUnion.as_schema().adjacencies  # only scalars, objects, containers refine
assert "Schemas.HasSchema" not in [a.relation.name for a in plain.Schemas.OfRelation.as_schema().anonymous]  # not a type
assert plain.Schemas.OfRelation.as_schema().properties["uniques"].schema.element.element is not None
other = B.Store()
assert other.Schemas.OfObject.as_schema() is not OfObject and other.Schemas.OfInt.as_schema() is not plain.Schemas.OfInt.as_schema()

## BLD-02 · Building in a store: a callback creates, an object and a callback update, an object alone clones

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    assert store.registered("crm.Person") is Person and store.registered("crm.Knows") is Knows
    assert store.registered("Schemas.OfObject") is store.Schemas.OfObject.as_schema()  # the store holds its plain schemas of schemas
    assert Person.validate() == [] and Knows.validate() == []
    bob = store.crm.Person(lambda c: c.given_name("Bob").age(41))
    ada = store.crm.Person(lambda c: (
        c
        .given_name("Ada")
        .age(36)
        .acquaintances(lambda adv: adv.insert(lambda env: env.known(bob).since(2020)))
    ))
    assert [str(e.known.given_name) for e in ada.acquaintances] == ["Bob"]
    store.crm.Person(ada, lambda c: c.age(37))
    assert int(ada.age) == 37
    twin = store.crm.Person(ada)
    assert twin is not ada and str(twin.given_name) == "Ada"
    assert repr(store.crm.Person) == "<crm.Person>"
    with raises(TypeError, match="takes a callback, an object and a callback, or an object"):
        store.crm.Person()
    with raises(AttributeError, match="no schema registered as 'crm.Nobody'"):
        store.crm.Nobody(lambda c: c)
    with raises(AttributeError):
        store._hidden
    with raises(AttributeError):
        store.crm._hidden

## BLD-03 · Paths: a named schema's is its name, a member's its owner's then its name; one with no name of its own has none, and one object alone links it

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")


def path(schema_object):
    """A schema object's path as plain data: from the root module, ".", by name (Object identity, Decided, 81)."""
    return Plain.ToPlain(store)(store.Paths.OfRoot.as_schema(), Paths.of(schema_object))


def step(name, i="schemas", then=None):
    return {"entry": {"i": i, "o": "", "properties": {"name": name}, "next": then or {"target": {}}}}


crm_person = {"singleton": ".", "next": step("crm", then=step("Person"))}
assert path(Person) == crm_person and store.held(Paths.of(Person)) is Person and store.held(crm_person) is Person
assert path(store.Schemas.HasSchema.as_schema()) == {"singleton": ".", "next": step("Schemas", then=step("HasSchema"))}
age = Person.properties["age"]
assert path(age) == {"singleton": ".", "next": step("crm", then=step("Person", then=step("age", "properties")))}
assert store.held(path(age)) is age  # a member's: its owner's, then the adjacency that owns it and its name there
assert store.held({"singleton": ".", "next": step("crm", then=step("Person", then=step("age", "nope")))}) is None
assert store.held({"singleton": "crm.Person", "next": {"target": {}}}) is None  # data's, not a schema object's
with store.transaction():
    with raises(ValueError, match="schema 'crm.Person' is already registered"):
        store.Schemas.OfObject(lambda o: o.name("crm.Person"))
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.name("crm.Human"))  # renamed: registered under its new name, at its new path
assert store.registered("crm.Human") is Person and "crm.Person" not in store.names() and store.held(crm_person) is None
assert path(Person) == {"singleton": ".", "next": step("crm", then=step("Human"))}
with store.transaction():
    assert store.crm.Human(lambda c: c.given_name("Cy")).as_schema() is Person
with store.transaction():
    with raises(ValueError, match="the path 'crm.Human' names another schema object already: a module and a schema, or two "
                                  "schemas, do not share a path"):
        store.Schemas.OfObject(lambda o: o.name("crm.Human.v2"))  # its module would be at the schema's path
# A schema object with no name of its own has no path, nor has one no store holds; one object alone links it.
with store.transaction():
    Tags = store.Schemas.OfContainer(lambda c: c.index([Int]).element([Text]))
with raises(LookupError, match="a container schema with no name of its own, nor a member of one that has a name, has no "
                               "path: it is written nested in the one object that links it"):
    Paths.of(Tags)
with raises(LookupError, match="a property no store holds has no path: a schema object's path is in its store"):
    Paths.of(B.OfProperty.Builder().schema([Text]).create())
with store.transaction():
    Tagged = store.Schemas.OfObject(lambda o: o.name("crm.Tagged").properties([lambda t: t.name("tags").schema([Tags])]))
with raises(LookupError, match="has no path"):
    Paths.of(Tags)  # linked, but not owned: still none
with store.transaction():
    with raises(ValueError, match="a container schema with no name of its own is linked by one schema object alone, and another "
                                  "links it already: name it to share it"):
        store.Schemas.OfObject(lambda o: o.name("crm.Labelled").properties([lambda t: t.name("labels").schema([Tags])]))
with store.transaction():
    store.Schemas.OfProperty(Tagged.properties["tags"], lambda p: p.schema([Text]))  # unlinked: another may link it
with store.transaction():
    Labelled = store.Schemas.OfObject(lambda o: o.name("crm.Labelled").properties([lambda t: t.name("labels").schema([Tags])]))
assert Labelled.properties["labels"].schema is Tags
with store.transaction():
    twin = B.OfObject.Builder(Labelled, True, store).name("crm.Twin").clone()  # a clone has its own of what has no name
assert twin.properties["labels"].schema is not Tags and twin.properties["labels"].schema.element is Text

## BLD-04 · A schema's members: the peer shorthand, each item linked, or a callback naming the entry and writing its member

In [ ]:
store = B.Store()
with store.transaction():
    Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
    spare = store.Schemas.OfProperty(lambda p: p.schema([Int]))
    Contact = store.Schemas.OfObject(lambda o: (
        o
        .name("crm.Contact")
        .ref(True)
        .properties([
            lambda t: t.name("given_name").schema([Text]),
            lambda t: t.name("spare").property(spare).description("extra"),  # linked, then updated
        ])
    ))
    assert list(Contact.properties) == ["given_name", "spare"] and Contact.properties["spare"] is spare
    assert spare.description == "extra" and spare.name == "spare"
    with raises(ValueError, match="link the object before writing its members"):
        B.OfObject.Builder().properties([lambda t: t.description("x").property(spare)])
    Bare = B.OfObject.Builder().properties([lambda t: t.name("x").property]).create()  # writes nothing: an empty one
    assert list(Bare.properties) == ["x"] and Bare.properties["x"].schema is None
    # the callback form: an entry, its name, and its link
    Person = (
        B.OfObject.Builder()
        .name("crm.Person")
        .ref(True)
        .properties(lambda adv: adv.insert(lambda env: env.name("nickname").property(lambda p: p.schema([Text]))))
        .create()
    )
    assert Person.validate() == [] and Person.properties["nickname"].schema is Text
    with raises(ValueError, match="link 'property' is not set"):
        B.OfObject.Builder().properties(lambda adv: adv.insert(lambda env: env.name("x")))
    with raises(TypeError, match="an entry, or a callback taking one, is expected"):
        B.OfObject.Builder().properties(lambda adv: adv.insert(3))
    with raises(TypeError, match="'schemas' takes a mapping only where its relation has two links and one property"):
        B.OfAny.Builder().schemas({"n": Int})
    with raises(TypeError, match="link 'property' cannot link 3"):
        B.OfObject.Builder().properties([3])

## BLD-05 · Relations: links by name, their schemas following from adjacencies; `me` and `uniques` naming links and properties

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    assert list(Knows.entry.links) == ["knower", "known"] and Knows.uniques == (frozenset({"since"}),)
    assert Person.adjacencies["acquaintances"].me == "knower" and [a.me for a in Person.anonymous] == ["known"]
    for given, held in (("since", (frozenset({"since"}),)),
                        (["knower", "since"], (frozenset({"knower", "since"}),)),
                        ([["since"], ["knower", "known"]], (frozenset({"since"}), frozenset({"knower", "known"}))),
                        (["since", ["knower", "known"]], (frozenset({"since"}), frozenset({"knower", "known"}))),
                        (None, ())):
        assert B.OfRelation.Builder(Knows).uniques(given).clone().uniques == held
    builder = B.OfRelation.Builder(Knows).uniques([["since"], ["known", "knower"], ["elsewhere"]])
    assert builder.uniques._held() == [["since"], ["knower", "known"], ["elsewhere"]] and builder.uniques.get() is None
    assert B.OfRelation.Builder().uniques._held() is None
    clauses, dicts = [], []
    B.OfRelation.Builder(Knows).uniques.as_value(lambda v: v.as_container(lambda c: (
        c.as_dict(lambda d: dicts.append(d)),
        c.as_entries(lambda it: [e.as_container(lambda n: n.as_entries(
            lambda names: clauses.append([x._held() for _, x in names]))) for _, e in it]),
    )))
    assert clauses == [["since"]] and dicts == []  # by position: as_dict is not called
    widened = B.OfRelation.Builder(Knows)
    widened.uniques.as_value(lambda v: v.as_container(lambda c: c.at(0).as_container(lambda n: n.at(1).as_scalar("knower"))))
    assert widened.clone().uniques == (frozenset({"since", "knower"}),)
    widened.uniques.as_value(lambda v: v.as_container(lambda c: c.remove(0)))
    assert widened.uniques._held() is None
    Unrelated = (
        B.OfAdjacency.Builder(Person.adjacencies["acquaintances"])
        .relation(lambda adv: adv.remove(lambda env: env.relation(lambda r: r.uniques("since"))))
        .clone()
    )
    assert Unrelated.relation is None  # the template's uniques agree with the relation's by value

## BLD-06 · Property shorthands: a value whole, of exactly the property's schema; a container's through its visitor

In [ ]:
for wrong in (lambda: B.OfObject.Builder().ref(1), lambda: B.OfObject.Builder().name(3)):
    with raises(TypeError, match="holds"):
        wrong()
# A property a map holds, as an extension may add: set whole, or visited as a container.
attributed = B.Store()
with attributed.transaction():  # a framework schema gains members in any transaction
    m = attributed
    m.Schemas.OfObject(m.Schemas.OfPattern, lambda o: o.properties([
        lambda t: t.name("attributes").schema([m.Schemas.OfContainer(lambda c: c.index([m.Schemas.OfStr]).element([
            m.Schemas.OfAny(lambda a: a.schemas([m.Schemas.OfStr, m.Schemas.OfInt, m.Schemas.OfFloat]))]))]),
        lambda t: t.name("default").schema([m.Schemas.OfAny(lambda a: a.schemas([m.Schemas.OfStr, m.Schemas.OfInt]))]),
    ]))
with attributed.transaction():
    called = attributed.Schemas.OfPattern(lambda m: m.name("call").attributes({"name": "knows"}).attributes({"arity": 2}).default(3))
assert B.OfPattern.Builder(called, False, attributed).default._held() == 3  # a value of any schema: as given
assert B.OfPattern.Builder(called, False, attributed).attributes._held() == {"arity": 2}  # set whole: nothing merges
with attributed.transaction():
    with raises(TypeError, match="holds a map"):
        attributed.Schemas.OfPattern(lambda m: m.attributes(["x"]))
with raises(TypeError, match="holds a list"):
    B.OfRelation.Builder().uniques._matched("since")
with raises(TypeError, match="holds a list"):
    B.OfRelation.Builder().uniques._matched(3)
pattern = B.OfPattern.Builder(None, True, attributed).name("call")  # in the store, which holds its plain schema
pattern.attributes(lambda p: p.as_value(lambda v: v.as_container(lambda c: c.at("name").as_scalar("knows"))))
pattern.attributes.as_value(lambda v: v.as_container(lambda c: c.at("arity").as_scalar(lambda s: s.set(2))))
seen = {}
pattern.attributes.as_value(lambda v: v.as_container(lambda c: (
    seen.update(length=len(c), schema=c.as_schema()),
    c.as_dict(lambda d: seen.update(keys=list(d))),
    c.as_entries(lambda it: seen.update(values=[e._held() for _, e in it])),
    c.as_ordered(lambda it: seen.update(ordered=True)),
)))
assert seen["length"] == 2 and seen["keys"] == ["name", "arity"] and seen["values"] == ["knows", 2]
assert "ordered" not in seen and seen["schema"] is attributed.Schemas.OfPattern.as_schema().properties["attributes"].schema
pattern.attributes.as_value(lambda v: v.as_container(lambda c: c.remove("arity")))
pattern.attributes.as_value(lambda v: v.as_container({"name": "calls"}))
pattern.attributes.as_value(lambda v: v.as_container([["name", "calls"]]))  # a map given as its entries
assert pattern.attributes._held() == {"name": "calls"}
pattern.attributes.as_value(lambda v: v.clear())
assert pattern.attributes._held() is None
pattern.attributes.as_value(lambda v: v.as_container(lambda c: c))  # selecting the kind writes it, empty
assert pattern.attributes._held() == {}
pattern.name.as_value(lambda v: (v.as_scalar(lambda s: (s.set("let"), s.get(), s.as_schema())), v.as_schema()))
assert str(pattern.name) == "let" and pattern.name.get() == "let"
pattern.name.as_value(lambda v: v.as_scalar(lambda s: s.clear()))
assert pattern.name.get() is None
pattern.name.as_scalar("var").as_value(lambda v: v.clear())
assert pattern.name.get() is None
with raises(TypeError, match="holds a scalar"):
    pattern.name.as_value(lambda v: v.as_object({}))
with raises(TypeError, match="holds a container"):
    pattern.attributes.as_value(lambda v: v.as_scalar(1))
reasons = []
pattern.name.as_object(lambda o: reasons.append(o.get()))
pattern.attributes.as_value(lambda v: v.as_scalar(lambda s: reasons.append(s.get())))
assert reasons == [None, None] and repr(pattern.name) == "<property 'name'>"
# A template agrees with a member's map by value; an OfAny's unnamed types take identities their path derives.
tagged = B.Store()
with tagged.transaction():
    tagged.Schemas.OfObject(tagged.Schemas.OfProperty, lambda o: o.properties([
        lambda t: t.name("tags").schema([tagged.Schemas.OfContainer(lambda c: c.index([tagged.Schemas.OfStr]).element([
            tagged.Schemas.OfStr]))])]))
with tagged.transaction():
    Tagged = tagged.Schemas.OfObject(lambda o: o.name("demo.Tagged").properties([
        lambda t: t.name("a").schema([tagged.Schemas.OfStr]).tags({"unit": "m"}),
        lambda t: t.name("b").schema([tagged.Schemas.OfStr]).tags({"unit": "s"}),
        lambda t: t.name("v").schema([tagged.Schemas.OfAny(lambda a: a.schemas([
            tagged.Schemas.OfContainer(lambda c: c.index([tagged.Schemas.OfInt]).element([tagged.Schemas.OfStr]))]))]),
    ]))
with tagged.transaction():
    tagged.Schemas.OfObject(Tagged, lambda o: o.properties(lambda adv: adv.remove(lambda env: env.property(lambda p: p.tags({"unit": "s"})))))
assert list(Tagged.properties) == ["a", "v"]
assert tagged.held(Paths.of(Tagged.properties["v"])) is Tagged.properties["v"]  # its path its owner's, then its name

## BLD-07 · Adjacency shorthands: one entry replaced where the relation allows one; `insert` does not replace

In [ ]:
Names = (
    B.OfContainer.Builder()
    .index([Int])
    .element([Text])
    .layout("dense")
    .create()
)
assert Names.validate() == [] and Names.arrangement == "dense"
Keyed = B.OfContainer.Builder(Names).index([Text]).clone()
assert Keyed.index is Text and Names.index is Int
with raises(ValueError, match="remove it before inserting another"):
    B.OfContainer.Builder(Names).index(lambda adv: adv.insert(lambda env: env.index(Text)))
B.OfContainer.Builder(Names).index(lambda adv: adv.insert(lambda env: env.index(Int)))  # held already: elided
with raises(TypeError, match="links an object that exists"):
    B.OfProperty.Builder().schema(lambda adv: adv.insert(lambda env: env.schema(lambda s: s)))
with raises(TypeError, match="links an object that exists: link it, not its members"):
    B.OfProperty.Builder().schema([lambda t: t.description("x")])
with raises(TypeError, match="link 'schema' cannot link 3"):
    B.OfProperty.Builder().schema(lambda adv: adv.insert(lambda env: env.schema(3)))
cleared = B.OfContainer.Builder(Names).index(lambda adv: adv.remove(lambda env: env.index(Int))).clone()
assert cleared.index is None

## BLD-08 · Reading back: members are child visitors; a link reads its referent, read-only

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    builder = B.OfObject.Builder(Person, True, store)  # in the store, which holds its plain schema
    assert builder.name == "crm.Person" and builder.ref.get() is True and builder.name.as_schema().name == "name"
    assert [str(env.name) for env in builder.properties] == ["given_name", "age"]
    first = next(iter(builder.properties))
    assert first.name.as_schema() is store.Schemas.HasProperty.as_schema().entry.properties["name"] and first.as_schema() is store.Schemas.HasProperty.as_schema().entry
    assert first.property.as_schema() is store.Schemas.OfProperty.as_schema()
    assert first.property.schema.as_schema().relation is store.Schemas.HasSchema.as_schema() and [p.as_schema().name for p in first.property.as_properties] == []
    assert repr(first) == "<entry of 'properties'>" and repr(first.property) == "<link 'property'>"
    assert repr(builder.properties) == "<adjacency 'properties'>" and repr(builder) == "<builder of an object schema>"
    links = []
    first.property.as_adjacencies(lambda adv: links.append(adv.as_schema().name))
    assert links == ["schema"]
    assert builder.as_schema() is store.Schemas.OfObject.as_schema() and builder.properties.as_schema() is store.Schemas.OfObject.as_schema().adjacencies["properties"]
    assert [p.as_schema().name for p in builder.as_properties] == ["name", "ref"]
    seen = []
    builder.as_adjacencies(lambda adv: seen.append(adv.as_schema().name))
    assert seen == ["parameters", "adjacencies", "properties", "refines"]
    iterated = []
    builder.properties.as_iterable(lambda it: iterated.extend(str(e.name) for e in it))
    assert iterated == ["given_name", "age"]
    with raises(ReadOnlyError, match="read-only"):
        first.property.description("x")
    with raises(ReadOnlyError):
        builder.nonesuch("x")
    assert first.nonesuch.get() is None and first.property._target() is not None
    empty = B.OfObject.Builder().properties(lambda adv: adv.insert(lambda env: (env.property.as_schema(), env.property(lambda p: p))))
    assert B.OfAny.Builder(None, True, store).schemas.as_schema() is store.Schemas.OfAny.as_schema().adjacencies["schemas"]
    detached = B.OfProperty.Builder()
    detached.__class__._meta = "Nonesuch"
    assert detached.description.as_schema() is None and detached.schema.as_schema() is None
    detached.__class__._meta = "OfProperty"
    B.OfObject.Builder._meta = "Nonesuch"
    unschemed = []
    B.OfObject.Builder().properties([lambda t: (unschemed.extend([t.as_schema(), t.name.as_schema()]), t)[1]])
    B.OfObject.Builder._meta = "OfObject"
    assert unschemed == [None, None]
    shared = B.OfObject.Builder().properties(lambda adv: adv.insert(lambda env: env.property(first.property))).create()
    assert shared.properties["given_name"] is Person.properties["given_name"]  # linked through the link given
    with raises(AttributeError):
        builder._hidden
    with raises(AttributeError):
        first._hidden
    with raises(AttributeError):
        first.property._hidden
    with raises(TypeError, match="is given"):
        B.OfObject.Builder(Text)

## BLD-09 · Editing through an entry: its member updated, or renamed; removing by entry or by template

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
age = Person.properties["age"]
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.properties(lambda adv: adv.as_iterable(
        lambda it: (next(it), next(it).property(lambda p: p.description("years"))))))
assert Person.properties["age"] is age and age.description == "years"
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.properties(lambda adv: adv.as_iterable(
        lambda it: (next(it), next(it).property(lambda p: p)))))  # writes nothing: nothing changes
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.properties(lambda adv: adv.as_iterable(lambda it: next(it).name("first_name"))))
assert list(Person.properties) == ["first_name", "age"] and Person.validate() == []
with store.transaction():
    store.Schemas.OfObject(Person, lambda o: o.properties(lambda adv: adv.as_iterable(lambda it: next(it).name.as_scalar("given"))))
assert list(Person.properties) == ["given", "age"]
with raises(TypeError, match="a name is non-empty text"):
    B.OfObject.Builder(Person).properties(lambda adv: adv.as_iterable(lambda it: next(it).name(3)))
with raises(TypeError, match="a name is non-empty text"):
    B.OfObject.Builder(Person).properties([lambda t: t.name("")])
Smaller = (
    B.OfObject.Builder(Person)
    .name("crm.Smaller")
    .properties(lambda adv: adv.remove(lambda env: env.name("age")))
    .clone()
)
assert list(Smaller.properties) == ["given"] and list(Person.properties) == ["given", "age"]
held = next(iter(B.OfObject.Builder(Person).properties))
Fewer = B.OfObject.Builder(Person).properties(lambda adv: adv.remove(held)).clone()
assert list(Fewer.properties) == ["age"]
Typed = B.OfObject.Builder(Person).properties(lambda adv: adv.remove(lambda env: env.property(lambda p: p.schema([Int])))).clone()
assert list(Typed.properties) == ["given"]
with raises(TypeError, match="an entry, or a callback taking one, is expected"):
    B.OfObject.Builder(Person).properties(lambda adv: adv.remove(3))
Copied = B.OfObject.Builder().name("demo.Copied").properties(B.OfObject.Builder(Person).properties).create()
assert list(Copied.properties) == ["given", "age"]
Again = B.OfObject.Builder().name("demo.Again").properties(lambda adv: adv.insert(held)).create()
assert list(Again.properties) == ["given"]

## BLD-10 · A clone owns its members, cloned with paths of their own, and its own of what has no name; inherited members stay shared

In [ ]:
store = B.Store()
Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
store.transaction(person)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
with store.transaction():
    Twin = store.Schemas.OfObject(B.OfObject.Builder(Person).name("crm.Twin").clone())
    assert Twin is not Person and list(Twin.properties) == ["given_name", "age"]
    assert Twin.properties["age"] is not Person.properties["age"] and Twin.properties["age"].schema is Int
    assert not Paths.same(Paths.of(Twin.properties["age"]), Paths.of(Person.properties["age"]))  # each its owner's
    Employee = store.Schemas.OfObject(lambda o: (
        o
        .name("crm.Employee")
        .refines([Person])
        .properties([lambda t: t.name("badge").schema([Text])])
    ))
    assert Employee.ref and list(Employee.properties) == ["given_name", "age", "badge"]
    Copy = B.OfObject.Builder(Employee).name("crm.Copy").clone()
    assert Copy.properties["age"] is Person.properties["age"] and Copy.properties["badge"] is not Employee.properties["badge"]
    assert Copy.adjacencies["acquaintances"] is Person.adjacencies["acquaintances"] and Copy.anonymous == Person.anonymous

## BLD-11 · Every kind: refinement, parameters, containers, unions, intersections, values of any schema, patterns

In [ ]:
store = B.Store()
with store.transaction():
    Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
    width = store.Schemas.OfParameter(lambda p: p.description("Its width"))
    Sized = store.Schemas.OfObject(lambda o: (
        o
        .name("demo.Sized")
        .parameters([lambda t: t.name("width").parameter(width)])
    ))
    assert Sized.parameters["width"] is width and width.name == "width"
    assert B.OfScalar.Builder(Int).name.get() == "Schemas.OfInt" and B.OfScalar.Builder().name.get() is None
    Names = store.Schemas.OfContainer(lambda c: c.name("demo.Names").index([Int]).element([Text]))
    Tags = store.Schemas.OfContainer(lambda c: c.name("demo.Tags").refines([Names]))
    assert Tags.index is Int and Tags.element is Text and Tags.validate() == []
    Narrow = store.Schemas.OfContainer(lambda c: c.name("demo.Narrow").refines([Names]).index([Int]).element([Text]))
    assert Narrow.index is Int and Narrow.element is Text and Narrow.validate() == []  # its own, given
    Shape = store.Schemas.OfUnion(lambda u: (
        u
        .name("geo.Shape")
        .flat(True)
        .branches([
            lambda t: t.name("count").schema([Int]),
            lambda t: t.name("label").schema([Text]),
        ])
    ))
    Stamped = store.Schemas.OfIntersection(lambda i: (
        i
        .name("geo.Stamped")
        .parts([
            lambda t: t.name("area").schema([store.Schemas.OfObject(lambda o: o.name("geo.Area").properties([lambda t: t.name("size").schema([Int])]))]),
            lambda t: t.name("when").schema([store.Schemas.OfObject(lambda o: o.name("geo.When").properties([lambda t: t.name("at").schema([Int])]))]),
        ])
    ))
    Either = store.Schemas.OfAny(lambda a: a.schemas([Int, Text]))
    Mutual = store.Schemas.OfPattern(lambda p: (
        p
        .name("crm.Mutual")
        .symbols([
            lambda t: t.name("a").schema([store.Schemas.OfObject(lambda o: o.name("crm.Who").ref(True))]),
            lambda t: t.name("b").schema([store.registered("crm.Who")]),
        ])
    ))
    for schema in (Sized, Shape, Stamped, Either, Mutual):
        assert schema.validate() == [], (schema, schema.validate())
    assert [b.name for b in Shape.branches] == ["count", "label"] and list(Mutual.symbols) == ["a", "b"]
    assert Either.schemas == (Int, Text)
    Linked = B.OfAdjacency.Builder().relation(lambda adv: adv.insert(lambda env: env.relation(lambda r: r.name("demo.R").links([lambda t: t.name("a"), lambda t: t.name("b")])))).me("a").create()
    assert Linked.relation.name == "demo.R" and Linked.me == "a"

## BLD-12 · Edges: reading through the shorthand's visitor, a scalar's conversions, finalizing in the wrong order

In [ ]:
scalar = B.OfScalar.Builder(None, True, store).name("demo.Name")  # in the store, which holds its plain schema
with raises(TypeError, match="holds a scalar"):
    scalar.name.as_value(lambda v: v.as_container({}))
with raises(TypeError, match="property 'name' holds a str, not an int"):
    scalar.name.as_value(lambda v: v.as_scalar(lambda s: int(s)))
Narrowed = B.OfContainer.Builder().name("demo.Narrowed").refines([Names]).create()
assert B.OfContainer.Builder(Narrowed).refines([Names]).clone().refines is Names  # held already: elided
seen = []
with store.transaction():
    B.OfObject.Builder(None, True, store).properties([lambda t: (
        seen.append((t.as_schema(), t.name.get())),
        t.name(lambda n: n.as_scalar("x")),
        seen.append((t.name.get(), t.name.as_schema().name)),
    )[1]])
assert seen == [(store.Schemas.HasProperty.as_schema().entry, None), ("x", "name")]
with raises(AttributeError):
    B.OfObject.Builder().properties([lambda t: t._hidden])
with raises(AttributeError):
    B.OfObject.Builder().properties([lambda t: t.name._hidden])
with raises(ValueError, match="link 'schema' is not set"):
    B.OfProperty.Builder().schema([lambda t: t])
with raises(ValueError, match="create() is only valid without a source instance"):
    B.OfObject.Builder(B.OfObject.Builder().create()).create()
twice = B.OfObject.Builder().name("demo.Twice")
first = twice.create()
second = twice.description("later").create()
assert first is not second and first.description is None and second.description == "later"
again = B.OfObject.Builder(first)
assert again.clone() is not again.clone() and first.description is None
with raises(ValueError, match="clone() is only valid with a source instance"):
    B.OfObject.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    B.OfObject.Builder().update()
with raises(TypeError):
    B.OfObject.Builder().create(1)

## BLD-13 · Proxies: a container property's value given whole; the peer shorthand on adjacencies, replacing where one entry is allowed

In [ ]:
store = B.Store()
with store.transaction():
    Text, Int, Bool = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema()
    P = store.Schemas.OfObject(lambda o: (
        o
        .name("crm.P")
        .ref(True)
        .properties([
            lambda t: t.name("n").schema([Text]),
            lambda t: t.name("tags").schema([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Text]))]),
            lambda t: t.name("marks").schema([store.Schemas.OfContainer(lambda c: c.index([Text]).element([Int]))]),
        ])
    ))
    K = store.Schemas.OfRelation(lambda r: (
        r
        .name("crm.K")
        .links([lambda t: t.name("a"), lambda t: t.name("b")])
        .properties([lambda t: t.name("since").schema([Int])])
    ))
    Best = store.Schemas.OfRelation(lambda r: r.name("crm.Best").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques("b"))
    Three = store.Schemas.OfRelation(lambda r: r.name("crm.Three").links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))
    store.Schemas.OfObject(P, lambda o: o.adjacencies([
        lambda t: t.name("knows").relation([K]).me("a"),
        lambda t: t.relation([K]).me("b"),
        lambda t: t.name("best").relation([Best]).me("a"),
        lambda t: t.relation([Best]).me("b"),
        lambda t: t.name("trio").relation([Three]).me("a"),
        lambda t: t.relation([Three]).me("b"),
        lambda t: t.relation([Three]).me("c"),
    ]))
    bob = store.crm.P(lambda c: c.n("Bob").tags(["x", "y"]).marks({"math": 3}))
    tags, marks = [], []
    bob.tags.as_value(lambda v: v.as_container(lambda c: c.as_entries(lambda it: [e.as_scalar(lambda s: tags.append(s.get())) for _, e in it])))
    bob.marks.as_value(lambda v: v.as_container(lambda c: c.as_entries(lambda it: [e.as_scalar(lambda s: marks.append((i, s.get()))) for i, e in it])))
    assert tags == ["x", "y"] and marks == [("math", 3)]
    ada = store.crm.P(lambda c: c.n("Ada").knows([bob, lambda t: t.since(2020).n("Cy")]))
    assert [str(e.b.n) for e in ada.knows] == ["Bob", "Cy"] and [e.since.get() for e in ada.knows] == [None, 2020]
    store.crm.P(ada, lambda c: c.knows([lambda t: t.b(bob).n("Robert")]))
    assert str(bob.n) == "Robert" and len(list(ada.knows)) == 3
    store.crm.P(ada, lambda c: c.best([bob]))
    store.crm.P(ada, lambda c: c.best([lambda t: t.n("Dee")]))
    assert [str(e.b.n) for e in ada.best] == ["Dee"]  # one entry allowed: replaced
    store.crm.P(ada, lambda c: c.knows(list(ada.knows)[:1]))  # the entry shorthand: inserted
    assert len(list(ada.knows)) == 4
    with raises(TypeError, match="have more than two links; give entries"):
        store.crm.P(ada, lambda c: c.trio([bob]))
    with raises(TypeError, match="more than one member of the entry takes <crm.P>: give a callback"):
        store.crm.P(ada, lambda c: c.knows({"x": bob}))  # the link, and the linked person's own 'knows', both take bob
    with raises(ValueError, match="link the object before writing its members"):
        store.crm.P(lambda c: c.knows([lambda t: t.n("Ed").b(bob)]))
    with raises(ValueError, match="link 'b' is not set"):
        store.crm.P(lambda c: c.knows([lambda t: t.since(1)]))
    seen = []
    store.crm.P(lambda c: c.knows([lambda t: (seen.append((t.as_schema() is K.entry, t.since.as_schema().name)), t.n("Fay"))[1]]))
    assert seen == [(True, "since")]
    with raises(AttributeError):
        store.crm.P(lambda c: c.knows([lambda t: t._hidden]))
    with raises(AttributeError):
        store.crm.P(lambda c: c.knows([lambda t: t.n._hidden]))
    chained = []
    store.crm.P(lambda c: c.knows([lambda t: (chained.append(t.n.as_schema().name), t.n("Gus"))[1]]))
    assert chained == ["n"]
    Trio = store.Schemas.OfRelation(lambda r: r.name("crm.Trio").links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))
    store.Schemas.OfObject(P, lambda o: o.adjacencies([
        lambda t: t.name("anon").relation([Trio]).me("a"),
        lambda t: t.relation([Trio]).me("b"),
        lambda t: t.relation([Trio]).me("c"),
    ]))
    with raises(TypeError, match="entries of crm.Trio have more than two links"):
        store.crm.P(ada, lambda c: c.anon([bob]))
    Lone = store.Schemas.OfRelation(lambda r: r.links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))
    store.Schemas.OfObject(P, lambda o: o.adjacencies([lambda t: t.name("lone").relation([Lone]).me("a")]))  # unnamed: one links it
    with raises(TypeError, match="entries of its relation have more than two links"):
        store.crm.P(ada, lambda c: c.lone([bob]))
    shown = []
    store.crm.P(lambda c: c.knows([lambda t: (shown.append(repr(t)), t.n("Jo"))[1]]))
    assert "_Them object" in shown[0]
    with raises(TypeError, match="takes a container of its schema"):
        store.crm.P(lambda c: c.tags("xy"))
    hal = store.crm.P(lambda c: c.knows([lambda t: t.b(lambda o: o.n("Hal")).n("Hank")]))  # built at the link, then written
    assert [str(e.b.n) for e in hal.knows] == ["Hank"]